# Baseline: Logistic Regression

This notebook builds a logistic regression baseline for comparison against the NCF model.

**Key design decisions to match NCF evaluation fairly:**
- Same `interactions.csv` dataset (positives + negatives)
- Same train/test split (`random_state=42`, `shuffle=True`, `test_size=0.2`)
- Same evaluation metrics: **Hit Rate@10** and **NDCG@10**
- Same evaluation protocol: per-user top-K ranking over all items (masking train items)

**Features used** (in place of embeddings):
- `cluster_id` (one-hot encoded) — customer segment from RFM clustering
- `Recency_scaled`, `Frequency_scaled`, `Monetary_scaled` — joined from RFM table per user
- `item_id` (one-hot encoded, truncated to top-N frequent items to keep it tractable)


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack, csr_matrix
import warnings
warnings.filterwarnings('ignore')

# Load datasets
interactions = pd.read_csv('data_processed/interactions.csv')
rfm = pd.read_csv('data_processed/rfm_clustered.csv')

print("Interactions shape:", interactions.shape)
print(interactions['purchased'].value_counts())
print()
print("Columns:", interactions.columns.tolist())
interactions.head()

Interactions shape: (2409660, 4)
purchased
0    1927728
1     481932
Name: count, dtype: int64

Columns: ['user_id', 'item_id', 'purchased', 'cluster_id']


,user_id,item_id,purchased,cluster_id
0,0,3104,1,3
1,0,1834,1,3
2,0,1212,1,3
3,0,219,1,3
4,0,1562,1,3


In [2]:
rfm

,Customer ID,Recency,Frequency,Monetary,Monetary_log,Frequency_log,Recency_scaled,Frequency_scaled,Monetary_scaled,Cluster,Segment,user_id
0,12346,326,12,77556.46,11.258774,2.564949,0.595584,1.254496,3.186625,3,Champions,0
1,12347,2,8,5633.32,8.636632,2.197225,-0.952279,0.800166,1.297127,3,Champions,1
2,12348,75,5,2019.40,7.611051,1.791759,-0.603532,0.299207,0.558100,2,Loyal,2
3,12349,19,4,4428.69,8.396085,1.609438,-0.871064,0.073946,1.123790,2,Loyal,3
4,12350,310,1,334.40,5.815324,0.693147,0.519146,-1.058146,-0.735888,0,Dormant,4
...,...,...,...,...,...,...,...,...,...,...,...,...
5873,18283,4,22,2736.65,7.914855,3.135494,-0.942724,1.959413,0.777020,3,Champions,5873
5874,18284,432,1,461.68,6.137036,0.693147,1.101983,-1.058146,-0.504065,0,Dormant,5874
5875,18285,661,1,427.00,6.059123,0.693147,2.195997,-1.058146,-0.560208,0,Dormant,5875
5876,18286,477,2,1296.43,7.168141,1.098612,1.316964,-0.557187,0.238942,0,Dormant,5876


## 1. Train/Test Split
Identical to the NCF split.

In [3]:
train, test = train_test_split(interactions, test_size=0.2, random_state=42, shuffle=True)

print(f"Train size: {len(train)}")
print(f"Test size:  {len(test)}")
print(f"Train purchased distribution:\n{train['purchased'].value_counts()}")
print(f"Test purchased distribution:\n{test['purchased'].value_counts()}")

Train size: 1927728
Test size:  481932
Train purchased distribution:
purchased
0    1542195
1     385533
Name: count, dtype: int64
Test purchased distribution:
purchased
0    385533
1     96399
Name: count, dtype: int64


In [4]:
train = train.merge(rfm[['user_id', 'Recency_scaled', 'Frequency_scaled', 'Monetary_scaled']], on='user_id', how='left')
test = test.merge(rfm[['user_id', 'Recency_scaled', 'Frequency_scaled', 'Monetary_scaled']], on='user_id', how='left')

## 2. Feature Engineering

Logistic regression can't learn embeddings, so we construct hand-crafted features:
- **RFM features** (Recency, Frequency, Monetary — already scaled) joined per user
- **cluster_id** one-hot encoded
- **item_id** one-hot encoded

We use sparse matrices to keep memory manageable.

In [5]:
def build_features(df, item_encoder, cluster_encoder):
    """
    Build a sparse feature matrix for logistic regression.
    Features: one-hot(item_id) + one-hot(cluster_id) + cluster-level RFM
    """
    # 1. One-hot encode item_id (sparse)
    item_ohe = item_encoder.transform(df[['item_id']])
    
    # 2. One-hot encode cluster_id (sparse)
    cluster_ohe = cluster_encoder.transform(df[['cluster_id']])
    
    # 3. Cluster-level RFM features (dense -> sparse)
    rfm_features = csr_matrix(df[['Recency_scaled', 'Frequency_scaled', 'Monetary_scaled']].values)
    return hstack([item_ohe, cluster_ohe, rfm_features])


# Fit encoders on TRAIN only (no data leakage)
item_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
item_encoder.fit(train[['item_id']])

cluster_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
cluster_encoder.fit(train[['cluster_id']])

X_train = build_features(train, item_encoder, cluster_encoder)
y_train = train['purchased'].values

X_test = build_features(test, item_encoder, cluster_encoder)
y_test = test['purchased'].values

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (1927728, 4638)
X_test shape:  (481932, 4638)


## 3. Train Logistic Regression

In [6]:
from sklearn.metrics import classification_report, roc_auc_score

lr = LogisticRegression(
    C=1.0,           # inverse regularisation strength
    max_iter=1000,
    solver='saga',   # fast solver for large sparse problems
    class_weight='balanced',  # counteract the 4:1 negative:positive imbalance
    random_state=42,
    n_jobs=-1
)

print("Training logistic regression...")
lr.fit(X_train, y_train)
print("Done.")

Training logistic regression...
Done.


In [7]:
# Binary classification report on test set
y_pred = lr.predict(X_test)
y_prob = lr.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Not purchased', 'Purchased']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

               precision    recall  f1-score   support

Not purchased       0.93      0.70      0.80    385533
    Purchased       0.39      0.78      0.52     96399

     accuracy                           0.71    481932
    macro avg       0.66      0.74      0.66    481932
 weighted avg       0.82      0.71      0.74    481932

ROC-AUC: 0.8131


## 4. Ranking Evaluation — Hit Rate@10 and NDCG@10

To compare fairly with NCF, we use the **same ranking protocol**:
1. For each user, score every item using the LR model
2. Mask items the user bought in training (set score to -inf)
3. Take Top-10
4. Check if any test items appear in the Top-10

The score is `predict_proba[:, 1]` (probability of purchase).

In [8]:
# Pre-compute train and test item sets per user
train_user_items = train[train['purchased'] == 1].groupby('user_id')['item_id'].apply(set).to_dict()
test_user_items  = test[test['purchased'] == 1].groupby('user_id')['item_id'].apply(set).to_dict()

num_users = interactions['user_id'].nunique()
num_items = interactions['item_id'].nunique()

print(f"num_users: {num_users}, num_items: {num_items}")

num_users: 5878, num_items: 4631


In [9]:
# Pre-build item feature matrix for all items (one row per item_id)
# We'll swap in each user's cluster at scoring time

# For efficiency: precompute item OHE for all items
all_items_df = pd.DataFrame({'item_id': range(num_items)})
item_ohe_all = item_encoder.transform(all_items_df[['item_id']])  # shape: (num_items, num_item_features)

print(f"Item OHE matrix shape: {item_ohe_all.shape}")

Item OHE matrix shape: (4631, 4631)


In [10]:
# Per-user lookup: user_id -> cluster_id and actual RFM values
user_cluster = interactions.drop_duplicates('user_id').set_index('user_id')['cluster_id'].to_dict()

# Build user_id -> (cluster_ohe, rfm array) cache using actual per-user RFM values
user_info = interactions.drop_duplicates('user_id')[['user_id', 'cluster_id']].merge(
    rfm[['user_id', 'Recency_scaled', 'Frequency_scaled', 'Monetary_scaled']],
    on='user_id',
    how='left'
)

user_feature_cache = {}
for _, row in user_info.iterrows():
    uid = int(row['user_id'])
    cluster_df = pd.DataFrame({'cluster_id': [row['cluster_id']]})
    cluster_ohe_row = cluster_encoder.transform(cluster_df[['cluster_id']])
    rfm_row = csr_matrix([[row['Recency_scaled'], row['Frequency_scaled'], row['Monetary_scaled']]])
    user_feature_cache[uid] = (cluster_ohe_row, rfm_row)

print(f"User feature cache built for {len(user_feature_cache)} users")


User feature cache built for 5878 users


In [11]:
from scipy.sparse import vstack as sp_vstack
import math

def score_user(user_id, top_k=10):
    """
    Score all items for a user using their actual RFM values.
    Returns top-K item indices after masking train items.
    """
    user_data = user_feature_cache.get(user_id)
    if user_data is None:
        return None

    cluster_ohe_row, rfm_row = user_data

    # Tile user features across all items: (1, F) -> (num_items, F)
    cluster_ohe_tiled = sp_vstack([cluster_ohe_row] * num_items)
    rfm_tiled         = sp_vstack([rfm_row] * num_items)

    # Full feature matrix: (num_items, num_features)
    X_user = hstack([item_ohe_all, cluster_ohe_tiled, rfm_tiled])

    # Score with LR
    scores = lr.predict_proba(X_user)[:, 1]  # shape: (num_items,)

    # Mask known train items
    for item_id in train_user_items.get(user_id, set()):
        if item_id < num_items:
            scores[item_id] = -np.inf

    # Top-K
    top_k_idx = np.argpartition(scores, -top_k)[-top_k:]
    top_k_idx = top_k_idx[np.argsort(scores[top_k_idx])[::-1]]  # sort descending
    return top_k_idx, scores


def calculate_hit_rate_lr(top_k=10):
    hits = 0
    for user_id in range(num_users):
        test_set = test_user_items.get(user_id, set())
        if not test_set:
            continue
        result = score_user(user_id, top_k)
        if result is None:
            continue
        top_k_idx, _ = result
        if len(set(top_k_idx) & test_set) > 0:
            hits += 1
    hit_rate = hits / num_users
    print(f"Hit Rate @{top_k}: {hit_rate:.4f}")
    return hit_rate


def calculate_ndcg_lr(top_k=10):
    total_ndcg = 0.0
    for user_id in range(num_users):
        test_set = test_user_items.get(user_id, set())
        if not test_set:
            continue
        result = score_user(user_id, top_k)
        if result is None:
            continue
        top_k_idx, _ = result
        user_ndcg = 0.0
        for rank, item_id in enumerate(top_k_idx):
            if item_id in test_set:
                user_ndcg += 1.0 / math.log2(rank + 2)
        total_ndcg += user_ndcg
    ndcg = total_ndcg / num_users
    print(f"NDCG @{top_k}: {ndcg:.4f}")
    return ndcg


print("Evaluating... (this may take a few minutes)")
hit_rate = calculate_hit_rate_lr(top_k=10)
ndcg     = calculate_ndcg_lr(top_k=10)


Evaluating... (this may take a few minutes)
Hit Rate @10: 0.3207
NDCG @10: 0.2372


## 5. Summary

| Metric | Logistic Regression (Baseline) | NCF |
|---|---|---|
| Hit Rate@10 | *(see above)* | *(from nfc_model.ipynb)* |
| NDCG@10 | *(see above)* | *(from nfc_model.ipynb)* |

**Why logistic regression is a reasonable but limited baseline:**
- It cannot personalise beyond the cluster level — all users in the same cluster get identical item scores
- It cannot capture user×item interaction effects (that's precisely what NCF's embeddings learn)
- One-hot item features make it essentially a popularity model within each cluster

A meaningful NCF improvement over this baseline demonstrates that learning individual user and item embeddings adds value beyond segment-level statistics.
